In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re 
import glob
from matplotlib.colors import ListedColormap
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from matplotlib import colors
import matplotlib.ticker as ticker

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.regression.mixed_linear_model import MixedLM
import statsmodels.stats.multitest as smm

In [2]:
# Read in data
cd8_t = pd.read_csv('../data/cd8_t_cells_cd8_low_cells_removed_clustering_results.csv')
cd8_t.head()

,Unnamed: 0,FlowSOM_metacluster_ 1,FlowSOM_metacluster_ 2,FlowSOM_metacluster_ 3,FlowSOM_metacluster_ 4,FlowSOM_metacluster_ 5,FlowSOM_metacluster_ 6,FlowSOM_metacluster_ 7,FlowSOM_metacluster_ 8,FlowSOM_metacluster_ 9,...,FlowSOM_metacluster_ 31,FlowSOM_metacluster_ 32,FlowSOM_metacluster_ 33,FlowSOM_metacluster_ 34,FlowSOM_metacluster_ 35,FlowSOM_metacluster_ 36,FlowSOM_metacluster_ 37,FlowSOM_metacluster_ 38,FlowSOM_metacluster_ 39,FlowSOM_metacluster_ 40
0,61201_001_C1_D8_T_Cell_Panel,112,125,201,16,9,555,5,213,979,...,591,770,186,309,615,19,86,171,489,19
1,61201_001_C7_D1_T_Cell_Panel,21,41,32,0,2,56,0,18,142,...,248,166,87,71,256,9,24,24,253,3
2,61201_001_C7_D22_T_Cell_Panel,12,16,20,1,2,23,0,20,73,...,151,117,17,58,142,4,2,20,26,3
3,61201_001_SPD_T_Cell_Panel,25,24,35,3,3,60,0,27,142,...,190,183,8,69,183,6,6,12,19,2
4,61201_002_C1_D1_T_Cell_Panel,1,3,37,242,2168,970,178,23,520,...,676,521,105,201,324,44,162,163,415,13


In [3]:
# Read in patient alias
patient_alias = pd.read_excel('../data/patient_alias_from_2025.xlsx', nrows= 40)
patient_alias_dict = dict(zip(patient_alias['PID'].astype(str), patient_alias['alias']))
patient_alias_dict

control_alias = pd.read_csv('../data/healthy_control.csv')
control_alias['key'] = control_alias['key'].str.replace('Healthy_BMA_67y_Male', 'Healthy_BMA_67Y_Male')

control_alias_dict = dict(zip(control_alias['key'].str.replace('_T_Cell_Panel.fcs', ''), control_alias['value'].str.replace('.fcs', '')))

In [4]:
# Rename the filenames
patient_id_dict = {}
for i in cd8_t['Unnamed: 0'].unique():
    if i.startswith('6'):
        patient_id_dict[i] = patient_alias_dict[i.split('_')[0] + i.split('_')[1]]#.replace('612500042', '61250004').replace('_T_Cell_Panel', '')]
    elif i.startswith('Healthy'):
        #x = i.split('_')[0] + '_' + i.split('_')[1] + '_' + i.split('_')[2]
        x = i.replace('_T_Cell_Panel', '')
        patient_id_dict[i] = control_alias_dict[x]

cd8_t['Patient_ID'] = cd8_t['Unnamed: 0'].map(patient_id_dict)

In [5]:
cd8_t

,Unnamed: 0,FlowSOM_metacluster_ 1,FlowSOM_metacluster_ 2,FlowSOM_metacluster_ 3,FlowSOM_metacluster_ 4,FlowSOM_metacluster_ 5,FlowSOM_metacluster_ 6,FlowSOM_metacluster_ 7,FlowSOM_metacluster_ 8,FlowSOM_metacluster_ 9,...,FlowSOM_metacluster_ 32,FlowSOM_metacluster_ 33,FlowSOM_metacluster_ 34,FlowSOM_metacluster_ 35,FlowSOM_metacluster_ 36,FlowSOM_metacluster_ 37,FlowSOM_metacluster_ 38,FlowSOM_metacluster_ 39,FlowSOM_metacluster_ 40,Patient_ID
0,61201_001_C1_D8_T_Cell_Panel,112,125,201,16,9,555,5,213,979,...,770,186,309,615,19,86,171,489,19,P08
1,61201_001_C7_D1_T_Cell_Panel,21,41,32,0,2,56,0,18,142,...,166,87,71,256,9,24,24,253,3,P08
2,61201_001_C7_D22_T_Cell_Panel,12,16,20,1,2,23,0,20,73,...,117,17,58,142,4,2,20,26,3,P08
3,61201_001_SPD_T_Cell_Panel,25,24,35,3,3,60,0,27,142,...,183,8,69,183,6,6,12,19,2,P08
4,61201_002_C1_D1_T_Cell_Panel,1,3,37,242,2168,970,178,23,520,...,521,105,201,324,44,162,163,415,13,P24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,Healthy_BMA_HSA2620_T_Cell_Panel,229,225,734,76,26,39,2,359,754,...,587,80,105,1858,43,76,18,123,37,Control_16
114,Healthy_BMA_HSA4362_T_Cell_Panel,23,10,19,1017,328,1369,1187,33,7,...,245,79,228,248,139,84,112,204,17,Control_10
115,Healthy_BMA_HSA4496_T_Cell_Panel,3,12,13,171,847,1028,873,24,108,...,616,177,316,4491,114,244,138,567,2,Control_4
116,Healthy_BMA_HSA4696_T_Cell_Panel,36,16,335,4,12,46,20,47,1778,...,76,7,41,243,5,10,19,72,2,Control_12


In [6]:
time_dict = {}
for i in cd8_t['Unnamed: 0']:
    if 'C1_D1' in i:
        time_dict[i] = 'C1_D1'
    elif 'C1_D8' in i:
        time_dict[i] = 'C1_D8'
    elif 'C7_D1' in i:
        time_dict[i] = 'C7_D1'
    elif 'C7_D22' in i:
        time_dict[i] = 'C7_D22'
    elif 'C12_D29' in i:
        time_dict[i] = 'C12_D29'
    elif 'C12_D29' in i:
        time_dict[i] = 'C12_D29'
    elif 'SPD_2' in i:
        time_dict[i] = 'SPD_2'
    elif 'SPD' in i:
        time_dict[i] = 'SPD'
    elif 'Healthy' in i:
        time_dict[i] = 'Control'

cd8_t['time'] = cd8_t['Unnamed: 0'].map(time_dict)

In [20]:
cd8_t = cd8_t.drop(['Unnamed: 0'], axis = 1)
cd8_t

,FlowSOM_metacluster_ 1,FlowSOM_metacluster_ 2,FlowSOM_metacluster_ 3,FlowSOM_metacluster_ 4,FlowSOM_metacluster_ 5,FlowSOM_metacluster_ 6,FlowSOM_metacluster_ 7,FlowSOM_metacluster_ 8,FlowSOM_metacluster_ 9,FlowSOM_metacluster_ 10,...,FlowSOM_metacluster_ 33,FlowSOM_metacluster_ 34,FlowSOM_metacluster_ 35,FlowSOM_metacluster_ 36,FlowSOM_metacluster_ 37,FlowSOM_metacluster_ 38,FlowSOM_metacluster_ 39,FlowSOM_metacluster_ 40,Patient_ID,time
0,112,125,201,16,9,555,5,213,979,103,...,186,309,615,19,86,171,489,19,P08,C1_D8
1,21,41,32,0,2,56,0,18,142,12,...,87,71,256,9,24,24,253,3,P08,C7_D1
2,12,16,20,1,2,23,0,20,73,10,...,17,58,142,4,2,20,26,3,P08,C7_D22
3,25,24,35,3,3,60,0,27,142,11,...,8,69,183,6,6,12,19,2,P08,SPD
4,1,3,37,242,2168,970,178,23,520,14,...,105,201,324,44,162,163,415,13,P24,C1_D1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,229,225,734,76,26,39,2,359,754,145,...,80,105,1858,43,76,18,123,37,Control_16,Control
114,23,10,19,1017,328,1369,1187,33,7,1,...,79,228,248,139,84,112,204,17,Control_10,Control
115,3,12,13,171,847,1028,873,24,108,89,...,177,316,4491,114,244,138,567,2,Control_4,Control
116,36,16,335,4,12,46,20,47,1778,79,...,7,41,243,5,10,19,72,2,Control_12,Control


In [22]:
cd8_t.to_csv('../data/cd8_t_cells_cd8_low_cells_removed_clustering_results_patient_updated.csv')

In [18]:
cd8_t

,Unnamed: 0,FlowSOM_metacluster_ 1,FlowSOM_metacluster_ 2,FlowSOM_metacluster_ 3,FlowSOM_metacluster_ 4,FlowSOM_metacluster_ 5,FlowSOM_metacluster_ 6,FlowSOM_metacluster_ 7,FlowSOM_metacluster_ 8,FlowSOM_metacluster_ 9,...,FlowSOM_metacluster_ 33,FlowSOM_metacluster_ 34,FlowSOM_metacluster_ 35,FlowSOM_metacluster_ 36,FlowSOM_metacluster_ 37,FlowSOM_metacluster_ 38,FlowSOM_metacluster_ 39,FlowSOM_metacluster_ 40,Patient_ID,time
0,61201_001_C1_D8_T_Cell_Panel,112,125,201,16,9,555,5,213,979,...,186,309,615,19,86,171,489,19,P08,C1_D8
1,61201_001_C7_D1_T_Cell_Panel,21,41,32,0,2,56,0,18,142,...,87,71,256,9,24,24,253,3,P08,C7_D1
2,61201_001_C7_D22_T_Cell_Panel,12,16,20,1,2,23,0,20,73,...,17,58,142,4,2,20,26,3,P08,C7_D22
3,61201_001_SPD_T_Cell_Panel,25,24,35,3,3,60,0,27,142,...,8,69,183,6,6,12,19,2,P08,SPD
4,61201_002_C1_D1_T_Cell_Panel,1,3,37,242,2168,970,178,23,520,...,105,201,324,44,162,163,415,13,P24,C1_D1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
113,Healthy_BMA_HSA2620_T_Cell_Panel,229,225,734,76,26,39,2,359,754,...,80,105,1858,43,76,18,123,37,Control_16,Control
114,Healthy_BMA_HSA4362_T_Cell_Panel,23,10,19,1017,328,1369,1187,33,7,...,79,228,248,139,84,112,204,17,Control_10,Control
115,Healthy_BMA_HSA4496_T_Cell_Panel,3,12,13,171,847,1028,873,24,108,...,177,316,4491,114,244,138,567,2,Control_4,Control
116,Healthy_BMA_HSA4696_T_Cell_Panel,36,16,335,4,12,46,20,47,1778,...,7,41,243,5,10,19,72,2,Control_12,Control
